Configuração Inicial e Imports

In [1]:
import os
import re
from pathlib import Path
import pandas as pd

caminho_projeto = Path.cwd()
if caminho_projeto.name == "Notebook_limpeza_dados":
    caminho_projeto = caminho_projeto.parent

caminho_entrada = caminho_projeto / "Datasets" / "datasetRefrigerador.csv"
caminho_saida = caminho_projeto / "Datasets" / "refrigeradores_limpo.csv"

Leitura e Padronização do Arquivo

In [2]:
try:
    df = pd.read_csv(caminho_entrada, encoding='utf-8')
    if len(df.columns) == 1:
        df = pd.read_csv(caminho_entrada, encoding='utf-8', sep=';')
except:
    df = pd.read_csv(caminho_entrada, encoding='latin1', sep=';')

df.columns = df.columns.str.replace('ï»¿', '').str.strip()

Limpeza e Identidade Base

In [3]:
df = df.dropna(subset=['Marca', 'Modelo']).drop_duplicates()

Unificação das Colunas de Consumo Mensal

In [4]:
if 'Consumo Médio de Energia Mensal (kWh/mês)' in df.columns and 'Consumo de Energia Mensal (kWh/mês)' in df.columns:
    df['Consumo_Mensal_kWh'] = df['Consumo Médio de Energia Mensal (kWh/mês)'].fillna(df['Consumo de Energia Mensal (kWh/mês)'])
elif 'Consumo Médio de Energia Mensal (kWh/mês)' in df.columns:
    df['Consumo_Mensal_kWh'] = df['Consumo Médio de Energia Mensal (kWh/mês)']
else:
    df['Consumo_Mensal_kWh'] = df['Consumo de Energia Mensal (kWh/mês)']

Resgate de Volume via Regex no Nome do Modelo

In [5]:
def extrair_volume(modelo):
    match = re.search(r'(\d{2,3})\s*(?:L|Lts|Litros)?\b', str(modelo), re.IGNORECASE)
    return float(match.group(1)) if match else None

df['Volume_Extraido'] = df['Modelo'].apply(extrair_volume)
df['Volume_Total_L'] = df['Volume Total (litros)'].fillna(df['Volume_Extraido'])

Imputação de Consumo por Média de Aparelhos Semelhantes

In [6]:
df['Consumo_Mensal_kWh'] = df.groupby(['Marca', 'Volume_Total_L'])['Consumo_Mensal_kWh'].transform(lambda x: x.fillna(x.mean()))

df = df.dropna(subset=['Consumo_Mensal_kWh']).copy()

Exportação e Deleção do Arquivo Original Sujo

In [7]:
df = df.drop(columns=['Volume_Extraido'])
df.to_csv(caminho_saida, index=False, encoding="utf-8")

if caminho_entrada.exists():
    os.remove(caminho_entrada)
    print(f"✓ Arquivo original deletado: {caminho_entrada.name}")

print(f"✓ Dataset limpo salvo em: {caminho_saida}")

✓ Arquivo original deletado: datasetRefrigerador.csv
✓ Dataset limpo salvo em: c:\Users\mathe\OneDrive\Área de Trabalho\Projeto TCS\Datasets\refrigeradores_limpo.csv
